In [1]:
# standardní knihovny Pythonu
import sys
import time
import re
from pathlib import Path

# knihovny třetích stran
import pandas as pd
import requests
from bs4 import BeautifulSoup

# nastavení root složky projektu pro import utility modulů
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# sdílené konstanty a společné schéma datasetu
from utils.constants import HEADERS
from utils.schema import FINAL_COLUMNS

# základní technické čištění textu
from utils.text_cleaning import (
    clean_base_text,
    join_text_parts
)

In [2]:
# zkratky tematických oblastí použitých pro sběr předmětů
SUBJECTS = [
    "ACCT", "ECON", "FINA", "MARK", "MGMT",
    "ISOM", "GBUS", "RMBI", "ENTR",
    "DASC", "COMP", "MATH"
]

# URL šablona katalogu předmětů HKUST
BASE_URL = "https://prog-crs.hkust.edu.hk/ugcourse/2025-26/{subject}"

In [3]:
# extrakce číselné hodnoty kreditů
def parse_credits(text):

    if not text:
        return None

    match = re.search(r"([\d.]+)", text)

    if match:
        return float(match.group(1))

    return None


# kontrola přítomnosti označení čínského jazyka výuky
def has_chinese_notation(text):

    if not text:
        return False

    return any(
        tag in text
        for tag in ["[PU]", "[CA]", "[C]"]
    )


# extrakce textového pole z detailu předmětu podle názvu sekce
def get_field(course_div, header_text):

    if not course_div:
        return None

    for row in course_div.find_all("div", class_="data-row-long"):

        header = row.find("div", class_="header")

        if not header:
            continue

        if header_text.lower() not in header.get_text(strip=True).lower():
            continue

        data = row.find("div", class_="data")

        if not data:
            return None

        items = data.find_all("li", class_="cilo")

        if items:

            output = []

            for item in items:

                seq = item.find("div", class_="cilo-seq")
                desc = item.find("div", class_="cilo-desc")

                if desc:

                    seq_text = (
                        seq.get_text(strip=True)
                        if seq else ""
                    )

                    output.append(
                        f"{seq_text} {desc.get_text(strip=True)}".strip()
                    )

            return join_text_parts(output)

        return clean_base_text(
            data.get_text(" ", strip=True)
        )

    return None

In [4]:
# stažení a parsování předmětů podle tematických oblastí
results = []
excluded = []

for subject in SUBJECTS:

    url = BASE_URL.format(subject=subject)

    print(f"Stahuji: {subject} | {url}")

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()
    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(response.text, "html.parser")
    courses = soup.find_all("li", class_="crse")

    subject_added = 0

    for course in courses:

        code_div = course.find("div", class_="crse-code")
        title_div = course.find("div", class_="crse-title")
        unit_div = course.find("div", class_="crse-unit")
        detail = course.find("div", class_="crse-detail")

        if not code_div or not title_div:
            continue

        course_code_text = clean_base_text(
            code_div.get_text(strip=True)
        )

        course_code = (
            course_code_text.replace(" ", "")
            if not pd.isna(course_code_text)
            else pd.NA
        )

        course_title = clean_base_text(
            title_div.get_text(strip=True)
        )

        credits_raw = (
            clean_base_text(unit_div.get_text(strip=True))
            if unit_div
            else pd.NA
        )

        credits = parse_credits(credits_raw)

        full_course_text = course.get_text(" ", strip=True)

        if credits == 0:

            excluded.append({
                "subject": subject,
                "course_code": course_code,
                "course_title": course_title,
                "reason": "0 credits"
            })

            continue

        if has_chinese_notation(full_course_text):

            excluded.append({
                "subject": subject,
                "course_code": course_code,
                "course_title": course_title,
                "reason": "Chinese language/material notation"
            })

            continue

        description = get_field(detail, "Description")

        learning_outcomes = get_field(
            detail,
            "Intended Learning Outcomes"
        )

        if pd.isna(description):

            excluded.append({
                "subject": subject,
                "course_code": course_code,
                "course_title": course_title,
                "reason": "missing description"
            })

            continue

        results.append({
            "course_title": course_title,
            "course_code": course_code,
            "university": "The Hong Kong University of Science and Technology",
            "country": "Hong Kong",
            "source_url": url,
            "academic_year": "2025-26",
            "semester": pd.NA,
            "semester_normalized": pd.NA,
            "study_level": "undergraduate",
            "credits": credits,
            "credit_system": "HKUST credits",

            # orientační převod podle doporučení OZS:
            # 1 HKUST credit = 2 ECTS
            "ects_credits": credits * 2 if credits is not None else pd.NA,

            "aims": description,
            "course_content": pd.NA,
            "learning_outcomes": learning_outcomes,
            "assessment_methods": pd.NA,
            "literature": pd.NA,
        })

        subject_added += 1

    print(f"  přidáno předmětů: {subject_added}")

    time.sleep(1)

Stahuji: ACCT | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ACCT
  přidáno předmětů: 21
Stahuji: ECON | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ECON
  přidáno předmětů: 38
Stahuji: FINA | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/FINA
  přidáno předmětů: 25
Stahuji: MARK | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/MARK
  přidáno předmětů: 15
Stahuji: MGMT | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/MGMT
  přidáno předmětů: 23
Stahuji: ISOM | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ISOM
  přidáno předmětů: 61
Stahuji: GBUS | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/GBUS
  přidáno předmětů: 11
Stahuji: RMBI | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/RMBI
  přidáno předmětů: 11
Stahuji: ENTR | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/ENTR
  přidáno předmětů: 11
Stahuji: DASC | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/DASC
  přidáno předmětů: 13
Stahuji: COMP | https://prog-crs.hkust.edu.hk/ugcourse/2025-26/COMP
  přidáno předmětů: 65

In [11]:
# vytvoření dataframe s naparsovanými předměty
df = pd.DataFrame(results)

# přehled vyřazených předmětů
excluded_df = pd.DataFrame(excluded)

# seřazení sloupců podle společného schématu datasetu
df = df[FINAL_COLUMNS]

print("Celkem získaných předmětů:", len(df))

Celkem získaných předmětů: 382


In [12]:
# export naparsovaných sylabů HKUST do interim vrstvy
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "hkust_syllabi_raw_parsed.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Uloženo {len(df)} kurzů do {OUTPUT_PATH.name}")

Uloženo 382 kurzů do hkust_syllabi_raw_parsed.csv
